# Week 24, Day 5: Edge cases and fixtures

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w024d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def chunk(items, size):
    return [items[i:i + size] for i in range(0, len(items), size)]

print(chunk([1, 2, 3, 4, 5], 2))
print(chunk([], 2))
print(chunk([1, 2], 5))
try:
    print(chunk([1, 2], 0))
except ValueError as e:
    print("ValueError:", e)

In [ ]:
import pytest

with open("app.py", "w") as f:
    f.write('''
import os

def mode():
    return os.environ.get("APP_MODE", "dev")

def report(items):
    for name, qty in items:
        print(f"{name}: {qty}")
    print(f"{len(items)} lines")

def save(path, items):
    path.write_text("\\n".join(f"{n},{q}" for n, q in items))
''')

with open("test_app.py", "w") as f:
    f.write('''
import pytest
from app import mode, report, save

@pytest.fixture
def items():
    return [("apples", 3), ("pears", 0)]

def test_default_mode(monkeypatch):
    monkeypatch.delenv("APP_MODE", raising=False)
    assert mode() == "dev"

def test_mode_from_env(monkeypatch):
    monkeypatch.setenv("APP_MODE", "live")
    assert mode() == "live"

def test_report(items, capsys):
    report(items)
    assert capsys.readouterr().out == "apples: 3\\npears: 0\\n2 lines\\n"

def test_save(items, tmp_path):
    path = tmp_path / "stock.csv"
    save(path, items)
    assert path.read_text() == "apples,3\\npears,0"
''')

pytest.main(["-v", "-p", "no:cacheprovider", "test_app.py"])

> **Key idea.** Run through an edge-case checklist: empty, one, boundaries, repeats, odd text, missing files, awkward dates. A fixture is a `@pytest.fixture` function that tests request by parameter name; each test gets a fresh result, and code after `yield` cleans up. Use `tmp_path` for files, `capsys` for printed output and `monkeypatch` for environment variables and swapped functions. Fixtures in `conftest.py` are shared by every test file in the folder.

### Exercise: Edge cases for chunk

`chunk(items, size)` in `chunks.py` splits a list into lists of `size` items (the last may be shorter), and raises `ValueError` if `size` is less than 1. Write tests that pass on the correct code and catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('chunks.py', 'w', encoding='utf-8') as f:
    f.write('def chunk(items, size):\n    if size < 1:\n        raise ValueError("size must be at least 1")\n    return [items[i:i + size] for i in range(0, len(items), size)]\n')

In [ ]:
import pytest
from chunks import chunk

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"chunks.py\").read()\nBROKEN = {\n    \"drops the short last chunk\": GOOD.replace(\"range(0, len(items), size)\", \"range(0, len(items) - size + 1, size)\"),\n    \"empty gives [[]]\": GOOD.replace(\"return [items\", \"return [[]] if not items else [items\"),\n    \"size 0 gives []\": GOOD.replace(\"size < 1\", \"size < 0\").replace(\"    return [items\", \"    if size == 0:\\n        return []\\n    return [items\"),\n    \"negative allowed\": GOOD.replace(\"    if size < 1:\\n        raise ValueError(\\\"size must be at least 1\\\")\\n\", \"    if size == 0:\\n        raise ValueError()\\n    size = abs(size)\\n\"),\n    \"overlapping\": GOOD.replace(\"range(0, len(items), size)\", \"range(0, len(items), max(1, size - 1))\"),\n}\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"chunks\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"chunks.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Your first fixture

`Basket` in `basket.py` is a class (you only need to use it: `Basket()` makes an empty basket). It has `add(name, price, qty=1)`, `remove(name)` (raises `KeyError` if it isn't there), `count()` (total quantity) and `total()`. Write a fixture `basket` that returns a basket with some items in it, and at least three tests that use it. Together they must catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('basket.py', 'w', encoding='utf-8') as f:
    f.write('class Basket:\n    def __init__(self):\n        self.items = {}\n\n    def add(self, name, price, qty=1):\n        old_price, old_qty = self.items.get(name, (price, 0))\n        self.items[name] = (price, old_qty + qty)\n\n    def remove(self, name):\n        del self.items[name]\n\n    def count(self):\n        return sum(q for _, q in self.items.values())\n\n    def total(self):\n        return round(sum(p * q for p, q in self.items.values()), 2)\n')

In [ ]:
import pytest
from basket import Basket

@pytest.fixture
def basket():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"basket.py\").read()\nBROKEN = {\n    \"adding again replaces\": GOOD.replace(\"old_qty + qty\", \"qty\"),\n    \"count is distinct items\": GOOD.replace(\"sum(q for _, q in self.items.values())\", \"len(self.items)\"),\n    \"total ignores qty\": GOOD.replace(\"p * q for p, q\", \"p for p, q\"),\n    \"remove ignores missing\": GOOD.replace(\"del self.items[name]\", \"self.items.pop(name, None)\"),\n    \"shared between baskets\": GOOD.replace(\"    def __init__(self):\\n        self.items = {}\\n\", \"    items = {}\\n\"),\n}\ndef test_fixture():\n    \"a fixture used by at least three passing tests\"\n    assert \"@pytest.fixture\" in source\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) >= 3 and not r[\"failed\"], f\"passed: {r['passed']}, failed: {r['failed']}\"\ndef test_catches():\n    \"your tests catch every broken version\"\n    results = against(source, \"basket\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"basket.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Test file code with tmp_path

`save_notes(path, notes)` in `notes.py` writes a list of notes to a file, one per line, and `load_notes(path)` reads them back, giving `[]` if the file doesn't exist and ignoring blank lines. Write tests with `tmp_path` that catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('notes.py', 'w', encoding='utf-8') as f:
    f.write('from pathlib import Path\n\ndef save_notes(path, notes):\n    Path(path).write_text("".join(n + "\\n" for n in notes), encoding="utf-8")\n\ndef load_notes(path):\n    path = Path(path)\n    if not path.exists():\n        return []\n    return [line for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]\n')

In [ ]:
from notes import save_notes, load_notes

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"notes.py\").read()\nBROKEN = {\n    \"missing file crashes\": GOOD.replace(\"    if not path.exists():\\n        return []\\n\", \"\"),\n    \"keeps blank lines\": GOOD.replace(\" if line.strip()]\", \"]\"),\n    \"appends instead of replacing\": GOOD.replace('    Path(path).write_text(\"\".join(n + \"\\\\n\" for n in notes), encoding=\"utf-8\")',\n        '    with open(path, \"a\", encoding=\"utf-8\") as f:\\n        f.write(\"\".join(n + \"\\\\n\" for n in notes))'),\n    \"loses the last note\": GOOD.replace(\"splitlines() if\", \"splitlines()[:-1] if\"),\n}\ndef test_style():\n    \"uses tmp_path\"\n    assert \"tmp_path\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"notes\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"notes.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Test printed output

`print_receipt(items)` in `receipt.py` prints one line per item, the name padded to 10 characters and the price with two decimal places, then a total line. For `[("tea", 2.5), ("biscuits", 1)]` it prints:

```
tea         2.50
biscuits    1.00
TOTAL       3.50
```

Write tests with `capsys` that catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('receipt.py', 'w', encoding='utf-8') as f:
    f.write('def print_receipt(items):\n    for name, price in items:\n        print(f"{name:<10}{price:>6.2f}")\n    print(f"{\'TOTAL\':<10}{sum(p for _, p in items):>6.2f}")\n')

In [ ]:
from receipt import print_receipt

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"receipt.py\").read()\nBROKEN = {\n    \"one decimal place\": GOOD.replace(\">6.2f}\\\")\\n    print\", \">6.1f}\\\")\\n    print\"),\n    \"no total\": \"\\n\".join(GOOD.splitlines()[:3]) + \"\\n\",\n    \"total on stderr\": \"import sys\\n\" + GOOD.replace(\"sum(p for _, p in items):>6.2f}\\\")\", \"sum(p for _, p in items):>6.2f}\\\", file=sys.stderr)\"),\n    \"total of first item only\": GOOD.replace(\"sum(p for _, p in items)\", \"(items[0][1] if items else 0)\"),\n}\ndef test_style():\n    \"uses capsys\"\n    assert \"capsys\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"receipt\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"receipt.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Change the settings for one test

`settings()` in `config.py` reads two environment variables: `APP_MODE` (default `"dev"`) and `APP_WORKERS` (a whole number, default 2, and `ValueError` if it isn't a whole number from 1 to 32). Write tests with `monkeypatch` that catch every broken version.

In [ ]:
# Run this cell first: it creates the files used below.
with open('config.py', 'w', encoding='utf-8') as f:
    f.write('import os\n\ndef settings():\n    mode = os.environ.get("APP_MODE", "dev")\n    workers = int(os.environ.get("APP_WORKERS", "2"))\n    if not 1 <= workers <= 32:\n        raise ValueError("APP_WORKERS must be from 1 to 32")\n    return {"mode": mode, "workers": workers}\n')

In [ ]:
import pytest
from config import settings

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"config.py\").read()\nBROKEN = {\n    \"default mode wrong\": GOOD.replace('\"APP_MODE\", \"dev\"', '\"APP_MODE\", \"live\"'),\n    \"ignores APP_MODE\": GOOD.replace('os.environ.get(\"APP_MODE\", \"dev\")', '\"dev\"'),\n    \"workers stay text\": GOOD.replace('int(os.environ.get(\"APP_WORKERS\", \"2\"))', 'os.environ.get(\"APP_WORKERS\", \"2\")').replace(\"    if not 1 <= workers <= 32:\\n        raise ValueError(\\\"APP_WORKERS must be from 1 to 32\\\")\\n\", \"\"),\n    \"no upper limit\": GOOD.replace(\"1 <= workers <= 32\", \"1 <= workers\"),\n    \"default workers 1\": GOOD.replace('\"APP_WORKERS\", \"2\"', '\"APP_WORKERS\", \"1\"'),\n}\ndef test_style():\n    \"uses monkeypatch\"\n    assert \"monkeypatch\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch every broken version\"\n    results = against(source, \"config\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"config.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Which fixture?

Name the built-in fixture for each job.

In [ ]:
a = ___   # check what a function printed
b = ___   # a fresh folder to write files in
c = ___   # pretend an environment variable is set, for one test
d = ___   # the file where fixtures shared by many test files go

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == \"capsys\" and b == \"tmp_path\" and c == \"monkeypatch\" and d == \"conftest.py\"")

### Bug hunt: A shared list

`test_add` passes, but then `test_starts_with_three` fails, because the fixture hands every test the **same** list, and `test_add` changed it. Fix the fixture so each test gets its own fresh list.

In [ ]:
import pytest

COLOURS = ["red", "green", "blue"]

@pytest.fixture
def colours():
    return COLOURS

def test_add(colours):
    colours.append("pink")
    assert "pink" in colours

def test_starts_with_three(colours):
    assert len(colours) == 3

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_both():\n    \"both tests pass\"\n    r = pytest_run(source)\n    assert len(r[\"passed\"]) == 2 and not r[\"failed\"], f\"failing: {r['failed']}\"")

### Bug hunt: Forgot to ask

This test fails with a confusing error. It uses the fixture's name, but it never **asks** for the fixture. Fix it.

In [ ]:
import pytest

@pytest.fixture
def prices():
    return [2.0, 3.5, 4.5]

def test_total():
    assert sum(prices) == 10.0

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_passes():\n    \"the test passes\"\n    r = pytest_run(source)\n    assert r[\"passed\"] == [\"test_total\"], f\"failing: {r['failed']}\"")

### Puzzle: A fixture with clean-up

Arrange the lines into a fixture that opens a file for a test and closes it afterwards, and a test that uses it. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: test a to-do store

`TodoStore(path)` in `todos.py` keeps a to-do list in a JSON file. `add(text)` adds an item (stripping spaces; empty text raises `ValueError`) and returns its number, starting at 1; `done(number)` marks it done (`KeyError` for an unknown number); `pending()` returns the texts not yet done, in order. Every change is saved straight away, and a new `TodoStore` on the same path sees earlier items. A missing file means an empty list.

Write a fixture that gives each test a store in `tmp_path`, and tests that pass on the correct code and catch all **six** broken versions.

In [ ]:
# Run this cell first: it creates the files used below.
with open('todos.py', 'w', encoding='utf-8') as f:
    f.write('import json\nfrom pathlib import Path\n\nclass TodoStore:\n    def __init__(self, path):\n        self.path = Path(path)\n        self.items = json.loads(self.path.read_text()) if self.path.exists() else []\n\n    def _save(self):\n        self.path.write_text(json.dumps(self.items))\n\n    def add(self, text):\n        text = text.strip()\n        if not text:\n            raise ValueError("empty to-do")\n        self.items.append({"text": text, "done": False})\n        self._save()\n        return len(self.items)\n\n    def done(self, number):\n        if not 1 <= number <= len(self.items):\n            raise KeyError(number)\n        self.items[number - 1]["done"] = True\n        self._save()\n\n    def pending(self):\n        return [i["text"] for i in self.items if not i["done"]]\n')

In [ ]:
import pytest
from todos import TodoStore

@pytest.fixture
def store(tmp_path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"todos.py\").read()\nBROKEN = {\n    \"not saved\": GOOD.replace(\"        self.items.append({\\\"text\\\": text, \\\"done\\\": False})\\n        self._save()\", \"        self.items.append({\\\"text\\\": text, \\\"done\\\": False})\"),\n    \"done not saved\": GOOD.replace(\"        self.items[number - 1][\\\"done\\\"] = True\\n        self._save()\", \"        self.items[number - 1][\\\"done\\\"] = True\"),\n    \"numbers from 0\": GOOD.replace(\"return len(self.items)\", \"return len(self.items) - 1\").replace(\"self.items[number - 1]\", \"self.items[number]\").replace(\"1 <= number <= len(self.items)\", \"0 <= number < len(self.items)\"),\n    \"no strip\": GOOD.replace(\"text = text.strip()\\n\", \"\").replace(\"if not text:\", \"if not text.strip():\"),\n    \"0 accepted\": GOOD.replace(\"1 <= number <= len(self.items)\", \"number <= len(self.items)\"),\n    \"empty allowed\": GOOD.replace(\"        if not text:\\n            raise ValueError(\\\"empty to-do\\\")\\n\", \"\"),\n}\ndef test_style():\n    \"a fixture using tmp_path\"\n    assert \"@pytest.fixture\" in source and \"tmp_path\" in source\ndef test_catches():\n    \"your tests pass on the correct code and catch all six broken versions\"\n    results = against(source, \"todos\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"todos.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

## Warm-up

The ordinary case is fine, and so are the empty list and the oversized chunk. But a size of 0 crashes inside `range` with a message about its step, not one about `chunk`. Nobody would find that by testing `[1, 2, 3, 4, 5]` with size 2. Edge cases are where you find out what a function really does.

## An edge-case checklist

Run through this list whenever you test something. Not every line applies every time; the point is not to forget one that does.

**Numbers:** 0; 1; negative; the boundaries of each rule and either side of them; very large; floats that don't add exactly (0.1 + 0.2); `int` where you expected `float` and the other way round.

**Collections:** empty; one item; two items (the smallest case with a "between"); repeats; already sorted, and in reverse; the item you want is first, last, or missing; very long.

**Text:** empty `""`; only spaces; spaces at the ends; mixed case; accents and other alphabets (`"Zoë"`, `"東京"`); a line break inside; punctuation; numbers written as text (`"07"`).

**Files:** missing; empty; one line; no line break at the end; a very large file; a path with spaces in it.

**Dates and times:** 29 February; the end of a month and of a year; midnight; a date given in another time zone; the hour the clocks change.

**Calls:** a function called twice (does the second call see leftovers from the first?); arguments of the wrong type; `None`.

Decide what **should** happen in each case before testing it. Often the answer is "raise a clear error", and that's a behaviour worth testing too.

## Fixtures

When several tests need the same setup, copying it into each one is tedious and easy to get wrong. A **fixture** is a function that prepares something for tests. A test asks for it by naming it as a parameter, and pytest calls the fixture and passes in the result:

The code being tested here, `Library`, is a **class**: a way to bundle data with the functions that work on it. You'll learn to write classes in Phase 3; for now you only need to **use** one, and that works just like the objects you already know. `Library()` makes a new, empty library, and you call its methods with a dot, as you do with a list's `append`.

```
import pytest
from library import Library

@pytest.fixture
def library():
    lib = Library()
    lib.add("Dune", copies=2)
    lib.add("Emma", copies=1)
    return lib

def test_borrow_reduces_copies(library):
    library.borrow("Emma")
    assert library.available("Emma") == 0

def test_cannot_borrow_missing_book(library):
    with pytest.raises(KeyError):
        library.borrow("Ulysses")
```

Key points:

- pytest runs the fixture **again for every test** that asks for it, so each test gets a fresh `Library`. The first test borrowing "Emma" doesn't affect the second. This is what keeps tests independent.
- A fixture can ask for other fixtures, by naming them as its own parameters.
- A test can ask for several fixtures.

**Cleaning up.** If a fixture needs to tidy up afterwards (close a connection, delete something), use `yield` instead of `return`. Code after the `yield` runs when the test finishes, whether it passed or failed:

```
@pytest.fixture
def connection():
    conn = open_connection()
    yield conn
    conn.close()
```

Fixtures normally run once per test. `@pytest.fixture(scope="module")` shares one result across a whole file, which is worth it only for slow setup that tests don't change.

## Built-in fixtures

pytest comes with fixtures you can ask for by name. Three are used all the time.

**`tmp_path`** gives each test its own new, empty temporary folder, as a `pathlib.Path`. Tests of file code should always use it, never real files in your project:

```
def test_save_and_load(tmp_path):
    path = tmp_path / "notes.txt"
    save_notes(path, ["milk", "eggs"])
    assert load_notes(path) == ["milk", "eggs"]

def test_missing_file(tmp_path):
    assert load_notes(tmp_path / "nothing.txt") == []
```

**`capsys`** captures what the code prints. `capsys.readouterr()` returns an object with `.out` and `.err`:

```
def test_greet_prints(capsys):
    greet("Sam")
    assert capsys.readouterr().out == "Hello, Sam!\n"
```

**`monkeypatch`** changes something for the length of one test, then puts it back:

- `monkeypatch.setenv("APP_MODE", "test")` and `monkeypatch.delenv("APP_MODE", raising=False)` for environment variables;
- `monkeypatch.setattr(module, "name", replacement)` to swap a function or value, such as replacing a function that fetches from the internet with one that returns fixed data;
- `monkeypatch.chdir(path)` to change the working folder.

Here they all are, run with real pytest:

Other built-in fixtures you may meet: `tmp_path_factory` (temporary folders shared across tests), `caplog` (captures `logging` messages) and `request` (information about the test being run). `pytest --fixtures` lists them all.

## Sharing fixtures

A fixture defined in a test file is available to the tests in that file. To share fixtures across many test files, put them in a file called **`conftest.py`** in the tests folder. pytest loads it automatically; you never import it.

```
tests/
├── conftest.py          # fixtures here are available to every test below
├── test_borrowing.py
└── test_returns.py
```

## Practice

## Quiz

## Daily challenge

## Before you go

Take the edge-case checklist and run it against something you wrote earlier in the course, such as the inventory or the log analyser. Write down which cases you'd never tested. Tomorrow is Forge: a bigger testing task that puts the whole week together.